In [31]:
import pandas as pd
xls = pd.ExcelFile("cashbook and bank_st.xlsx")
cashbook = pd.read_excel(xls, "CashBook")
bank_stmt = pd.read_excel(xls, "Bank Statement")

In [32]:
# Convert types
cashbook['Transaction Date'] = pd.to_datetime(cashbook['Transaction Date'], format="%Y.%m.%d", errors="coerce" )
bank_stmt['Transaction Date'] = pd.to_datetime(bank_stmt['Transaction Date'], format="%Y.%m.%d", errors="coerce" )

cashbook['Balance'] = cashbook['Balance'].astype(float)
bank_stmt['Balance'] = bank_stmt['Balance'].astype(float)

cashbook["Credit"] = pd.to_numeric(cashbook["Credit"], errors="coerce")
cashbook["Debit"] = pd.to_numeric(cashbook["Debit"], errors="coerce")

bank_stmt["Credit"] = pd.to_numeric(bank_stmt["Credit"], errors="coerce")
bank_stmt["Debit"] = pd.to_numeric(bank_stmt["Debit"], errors="coerce")

cashbook["Description"] = cashbook["Description"].astype(str).str.strip().str.lower()
bank_stmt["Description"] = bank_stmt["Description"].astype(str).str.strip().str.lower()

In [33]:
merged = pd.merge(
    bank_stmt, cashbook,
    on=["Transaction Date", "Description", "Credit", "Debit"],
    how="outer",
    indicator=True
)

matched = merged[merged["_merge"] == "both"]

# W‑1: Bank → Cashbook mismatches
w1 = merged[merged['_merge'] == 'left_only']

# W‑2: Cashbook → Bank mismatches
w2 = merged[merged['_merge'] == 'right_only']

In [34]:
summary = pd.DataFrame({
    "Total Cashbook Entries": [len(cashbook)],
    "Total Bank Entries": [len(bank_stmt)],
    "Unmatched Bank→Cashbook": [len(w1)],
    "Unmatched Cashbook→Bank": [len(w2)]
})

In [35]:
with pd.ExcelWriter("Dual_Ledger_Output.xlsx", engine="openpyxl") as writer:
    cashbook.to_excel(writer, sheet_name="Cashbook", index=False)
    bank_stmt.to_excel(writer, sheet_name="Bank Statement", index=False)
    matched.to_excel(writer, sheet_name="Matched", index=False)
    w1.to_excel(writer, sheet_name="W1 Bank Only", index=False)
    w2.to_excel(writer, sheet_name="W2 Cashbook Only", index=False)
    summary.to_excel(writer, sheet_name="Summary", index=False)